# 株価データ取得プログラム
このノートブックは、指定した銘柄の株価データを取得し、CSVファイルとしてダウンロードします。

## 1. 必要なライブラリのインストールと読み込み

In [ ]:
# 必要なライブラリをインストール
!pip install -q yfinance pandas matplotlib japanize-matplotlib

# ライブラリの読み込み
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from datetime import datetime, timedelta
from google.colab import files

try:
    import japanize_matplotlib
    print('✓ 日本語フォント設定完了')
except ImportError:
    print('⚠️ japanize-matplotlib が見つかりません。ラベルが英語になる場合があります。')


## 2. 株価データ取得の設定
ここで銘柄コード、期間を設定します

In [ ]:
# ===== ここを編集してください =====

# 銘柄コード（例）
# 日本株: "7203.T" (トヨタ自動車)、"6758.T" (ソニー)
# 米国株: "AAPL" (Apple)、"MSFT" (Microsoft)、"GOOGL" (Google)
ticker_symbol = "7203.T"  # ここに取得したい銘柄コードを入力

# 期間設定（デフォルト: 直近1年間）
_today = datetime.today()
end_date   = _today.strftime("%Y-%m-%d")              # 本日
start_date = (_today - timedelta(days=365)).strftime("%Y-%m-%d")  # 1年前

# ★ 期間を手動で指定したい場合は、下の2行のコメントを外して入力してください
# start_date = "2024-01-01"
# end_date   = "2025-01-01"

# データの間隔（"1d"=日次、"1wk"=週次、"1mo"=月次）
interval = "1d"

# ===== 入力値の検証 =====
import re

date_pattern = r'^\d{4}-\d{2}-\d{2}$'
if not re.match(date_pattern, start_date) or not re.match(date_pattern, end_date):
    raise ValueError("日付は YYYY-MM-DD 形式で入力してください")

try:
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt   = datetime.strptime(end_date,   "%Y-%m-%d")
    if start_dt >= end_dt:
        raise ValueError("開始日は終了日より前の日付にしてください")
except ValueError as e:
    raise ValueError(f"日付が無効です: {e}")

if not ticker_symbol or ticker_symbol.strip() == "":
    raise ValueError("銘柄コードを入力してください")

valid_intervals = ["1m","2m","5m","15m","30m","60m","90m","1h","1d","5d","1wk","1mo","3mo"]
if interval not in valid_intervals:
    raise ValueError(f"データ間隔は次のいずれかを指定してください: {', '.join(valid_intervals)}")

print(f"銘柄コード: {ticker_symbol}")
print(f"期間: {start_date} から {end_date}")
print(f"データ間隔: {interval}")
print("✓ 入力値の検証が完了しました")


## 3. 株価データの取得

In [ ]:
# 株価データを取得
print(f"\n{ticker_symbol} のデータを取得中...")

# yfinanceを使用してデータ取得（エラーハンドリング追加）
try:
    stock_data = yf.download(
        ticker_symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        progress=True
    )
except Exception as e:
    error_msg = f"❌ データ取得中にエラーが発生しました: {str(e)}\n"
    error_msg += "以下を確認してください:\n"
    error_msg += "  - 銘柄コードが正しいか\n"
    error_msg += "  - インターネット接続が正常か\n"
    error_msg += "  - 日付範囲が適切か"
    raise RuntimeError(error_msg)

# データが取得できたか確認（重要: 空の場合は処理を中断）
if stock_data.empty:
    error_msg = "⚠️ データが取得できませんでした。\n"
    error_msg += "以下を確認してください:\n"
    error_msg += f"  - 銘柄コード「{ticker_symbol}」が正しいか\n"
    error_msg += f"  - 期間「{start_date}」から「{end_date}」にデータが存在するか\n"
    error_msg += "  - 日本株の場合は末尾に「.T」が付いているか (例: 7203.T)\n"
    error_msg += "\n⚠️ これ以降のセルは実行しないでください（空のファイルがダウンロードされます）"
    raise ValueError(error_msg)

# データ取得成功
print(f"✓ データ取得完了: {len(stock_data)} 件のデータ")
print(f"✓ カラム数: {len(stock_data.columns)}")
print(f"✓ カラム名: {', '.join(map(str, stock_data.columns))}")
print("\n最初の5件:")
print(stock_data.head())
print("\n最後の5件:")
print(stock_data.tail())

## 4. データの整形と確認

In [ ]:
# データフレームの情報を表示
print("データの概要:")
print(stock_data.info())

print("\n基本統計量:")
print(stock_data.describe())

# カラム名を日本語に変更（オプション）
# 注意: yfinanceのカラム構造が変更された場合に対応
stock_data_jp = stock_data.copy()

# 日本語化フラグの初期化
columns_converted_to_japanese = False

# カラム数の検証
expected_columns = 6
if len(stock_data_jp.columns) != expected_columns:
    print(f"⚠️ 警告: 予期しないカラム数です（期待: {expected_columns}, 実際: {len(stock_data_jp.columns)}）")
    print(f"カラム名: {list(map(str, stock_data_jp.columns))}")
    print("日本語カラム名への変換をスキップします。")
else:
    # 期待されるカラム名を検証（文字列に変換して比較）
    expected_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    actual_cols = [str(col) for col in stock_data_jp.columns]
    
    if actual_cols == expected_cols:
        stock_data_jp.columns = ['始値', '高値', '安値', '終値', '調整後終値', '出来高']
        columns_converted_to_japanese = True  # フラグを設定
        print("✓ カラム名を日本語に変換しました")
        print("\n日本語カラム名でのプレビュー:")
        print(stock_data_jp.head())
    else:
        print(f"⚠️ 警告: カラム名が期待と異なります")
        print(f"期待: {expected_cols}")
        print(f"実際: {actual_cols}")
        print("日本語カラム名への変換をスキップします。元のカラム名を使用してください。")

## 4-2. 株価チャート（テクニカル分析付き）

取得した株価データを3つのパネルで可視化します。
- **上段**: 終値 ＋ 移動平均線（SMA20/50/200）＋ ボリンジャーバンド（±2σ）
- **中段**: 出来高（上昇日=緑、下落日=赤）
- **下段**: RSI(14) — 70超=過熱圏、30未満=売られ過ぎ圏

In [ ]:
# ----- テクニカル指標の計算 -----
_df = stock_data.copy()
if isinstance(_df.columns, pd.MultiIndex):
    _df.columns = _df.columns.get_level_values(0)

close  = _df['Close'].squeeze().astype(float)
open_  = _df['Open'].squeeze().astype(float)
volume = _df['Volume'].squeeze().astype(float)
dates  = _df.index

sma20  = close.rolling(20).mean()
sma50  = close.rolling(50).mean()
sma200 = close.rolling(200).mean()

bb_mid   = close.rolling(20).mean()
bb_std   = close.rolling(20).std()
bb_upper = bb_mid + 2 * bb_std
bb_lower = bb_mid - 2 * bb_std

_delta = close.diff()
_gain  = _delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
_loss  = (-_delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
rsi    = 100 - 100 / (1 + _gain / _loss.replace(0, np.nan))

# 出来高の色（上昇日=緑、下落日=赤）
vol_colors = ['#26a69a' if c >= o else '#ef5350'
              for c, o in zip(close.values, open_.values)]

# ----- プロット -----
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(3, 1, figure=fig, height_ratios=[3, 1, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

fig.suptitle(f"{ticker_symbol}  株価チャート  ({start_date} ～ {end_date})",
             fontsize=14, y=1.01)

# --- Panel 1: 終値 + SMA + ボリンジャーバンド ---
ax1.fill_between(dates, bb_lower, bb_upper, alpha=0.10, color='gray', label='BB ±2σ')
ax1.plot(dates, bb_upper, color='gray', linewidth=0.7, linestyle=':')
ax1.plot(dates, bb_lower, color='gray', linewidth=0.7, linestyle=':')
ax1.plot(dates, close,  label='終値',   color='#1565C0', linewidth=1.4, zorder=3)
if not sma20.isna().all():
    ax1.plot(dates, sma20,  label='SMA20',  color='#FF8F00', linewidth=1.1, linestyle='--')
if not sma50.isna().all():
    ax1.plot(dates, sma50,  label='SMA50',  color='#2E7D32', linewidth=1.1, linestyle='--')
if not sma200.isna().all():
    ax1.plot(dates, sma200, label='SMA200', color='#C62828', linewidth=1.1, linestyle='--')
ax1.set_ylabel('価格', fontsize=11)
ax1.legend(loc='upper left', fontsize=8, framealpha=0.7, ncol=3)
ax1.grid(True, alpha=0.25)
plt.setp(ax1.get_xticklabels(), visible=False)

# --- Panel 2: 出来高 ---
ax2.bar(dates, volume, color=vol_colors, alpha=0.80, width=0.8)
vol_ma5 = volume.rolling(5).mean()
ax2.plot(dates, vol_ma5, color='navy', linewidth=1.0, label='出来高 MA5')
ax2.set_ylabel('出来高', fontsize=11)
ax2.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1e8:.1f}億' if x >= 1e8
                      else (f'{x/1e6:.0f}M' if x >= 1e6 else f'{x/1e3:.0f}K')))
ax2.legend(loc='upper left', fontsize=8, framealpha=0.7)
ax2.grid(True, alpha=0.25)
plt.setp(ax2.get_xticklabels(), visible=False)

# --- Panel 3: RSI ---
ax3.plot(dates, rsi, color='#6A1B9A', linewidth=1.3, label='RSI(14)')
ax3.axhline(70, color='#ef5350', linestyle='--', linewidth=1.0, alpha=0.9)
ax3.axhline(50, color='gray',    linestyle=':',  linewidth=0.8, alpha=0.7)
ax3.axhline(30, color='#26a69a', linestyle='--', linewidth=1.0, alpha=0.9)
ax3.fill_between(dates, rsi, 70, where=(rsi >= 70), alpha=0.18, color='#ef5350', label='過熱圏')
ax3.fill_between(dates, rsi, 30, where=(rsi <= 30), alpha=0.18, color='#26a69a', label='売られ過ぎ圏')
ax3.text(dates[-1], 72, '過熱(70)', color='#ef5350', fontsize=8, ha='right')
ax3.text(dates[-1], 23, '売られ過ぎ(30)', color='#26a69a', fontsize=8, ha='right')
ax3.set_ylabel('RSI(14)', fontsize=11)
ax3.set_ylim(0, 100)
ax3.grid(True, alpha=0.25)

ax3.xaxis.set_major_locator(mdates.MonthLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))
fig.autofmt_xdate(rotation=40)

plt.tight_layout()
chart_file = f"{ticker_symbol}_chart.png"
plt.savefig(chart_file, dpi=130, bbox_inches='tight')
plt.show()
print(f"✓ チャートを保存しました: {chart_file}")


## 5. CSVファイルとしてダウンロード

In [ ]:
# ファイル名を生成
filename = f"{ticker_symbol}_{start_date}_to_{end_date}.csv"

# データの再確認（念のため）
if stock_data.empty:
    raise ValueError("❌ データが空です。このセルを実行する前にデータを取得してください。")

# CSVファイルとして保存（元のカラム名）
try:
    stock_data.to_csv(filename, encoding='utf-8-sig')  # utf-8-sigでExcelでも文字化けしない
    print(f"✓ CSVファイルを作成しました: {filename}")
    print(f"  - 行数: {len(stock_data)}")
    print(f"  - カラム数: {len(stock_data.columns)}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    print("\nダウンロードを開始します...")
    files.download(filename)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename}」は作成されていますが、ダウンロードできませんでした。")

## 6. (オプション) 日本語カラム名でもダウンロード

In [ ]:
# 日本語カラム名バージョンもダウンロードしたい場合

# stock_data_jpが存在するか確認
try:
    if stock_data_jp.empty:
        raise ValueError("データが空です")
except NameError:
    raise NameError("❌ 「stock_data_jp」が定義されていません。先にセル8を実行してください。")

# 日本語化フラグの確認
try:
    is_japanese = columns_converted_to_japanese
except NameError:
    # フラグが未定義の場合、カラム名から判定
    is_japanese = '始値' in [str(col) for col in stock_data_jp.columns]

# ファイル名を動的に決定
if is_japanese:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_日本語.csv"
else:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_copy.csv"
    print("⚠️ カラムが日本語化されていないため、ファイル名を「_copy.csv」としました")

# CSVファイルとして保存
try:
    stock_data_jp.to_csv(filename_jp, encoding='utf-8-sig')
    if is_japanese:
        print(f"✓ 日本語版CSVファイルを作成しました: {filename_jp}")
    else:
        print(f"✓ CSVファイルを作成しました: {filename_jp}")
    print(f"  - 行数: {len(stock_data_jp)}")
    print(f"  - カラム数: {len(stock_data_jp.columns)}")
    print(f"  - カラム名: {', '.join(map(str, stock_data_jp.columns))}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    files.download(filename_jp)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename_jp}」は作成されていますが、ダウンロードできませんでした。")

---
## 使い方のヒント

### 主要な日本株の銘柄コード例:
- トヨタ自動車: `7203.T`
- ソニーグループ: `6758.T`
- ソフトバンクグループ: `9984.T`
- キーエンス: `6861.T`
- 三菱UFJフィナンシャル・グループ: `8306.T`

### 主要な米国株の銘柄コード例:
- Apple: `AAPL`
- Microsoft: `MSFT`
- Google (Alphabet): `GOOGL`
- Amazon: `AMZN`
- Tesla: `TSLA`

### データ間隔の設定:
- `1d`: 日次データ
- `1wk`: 週次データ
- `1mo`: 月次データ
- `1h`: 時間足（直近60日間のみ）

### カラムの説明:
- **Open (始値)**: その期間の最初の取引価格
- **High (高値)**: その期間の最高価格
- **Low (安値)**: その期間の最低価格
- **Close (終値)**: その期間の最後の取引価格
- **Adj Close (調整後終値)**: 株式分割や配当を考慮した終値
- **Volume (出来高)**: その期間の取引量

---
## 7. 上昇予想銘柄のレコメンド機能

過去の株価推移（テクニカル指標）から、今後上昇が期待できる銘柄をスコアリングしてレコメンドします。

### 評価に使用するテクニカル指標
- **移動平均線 (SMA)**: 短期(20日)・中期(50日)・長期(200日)の関係から上昇トレンドを判定
- **ゴールデンクロス**: 短期SMAが中期SMAを上抜けると上昇シグナル
- **RSI (14日)**: 売られすぎ(30未満)からの反発、適度な水準(40〜60)を加点
- **モメンタム**: 直近20日・60日のリターン
- **ボリンジャーバンド**: バンド内の相対位置から過熱感を判定
- **出来高トレンド**: 直近の出来高増加で買い意欲を確認

### ⚠️ 免責事項
本機能は過去データに基づくテクニカル分析の自動化であり、将来の株価上昇を保証するものではありません。
投資判断は自己責任で行ってください。

In [ ]:
# ===== レコメンド対象の候補銘柄リスト =====
# ここに分析したい銘柄を追加してください。多いほどスクリーニングの幅が広がります。

candidate_tickers = {
    # 日本株（主要銘柄）
    "7203.T": "トヨタ自動車",
    "6758.T": "ソニーグループ",
    "9984.T": "ソフトバンクグループ",
    "6861.T": "キーエンス",
    "8306.T": "三菱UFJフィナンシャル",
    "9432.T": "NTT",
    "6098.T": "リクルートHD",
    "8035.T": "東京エレクトロン",
    "6501.T": "日立製作所",
    "7974.T": "任天堂",
    # 米国株（主要銘柄）
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "GOOGL": "Alphabet",
    "AMZN": "Amazon",
    "NVDA": "NVIDIA",
    "META": "Meta Platforms",
    "TSLA": "Tesla",
}

# レコメンドのパラメータ
lookback_days = 250          # 過去何日分のデータでテクニカル指標を計算するか（200日SMA算出のため250日推奨）
top_n = 5                    # 上位何銘柄を推奨表示するか
min_data_points = 60         # 最低でも必要な営業日数（これ未満は除外）

print(f"候補銘柄数: {len(candidate_tickers)}")
print(f"取得期間: 直近 {lookback_days} 日")
print(f"レコメンド上位: {top_n} 銘柄")


In [ ]:
# ===== テクニカル指標の計算とスコアリング =====
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta


def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """RSI（相対力指数）を計算"""
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def analyze_ticker(ticker: str, name: str, period_days: int) -> dict | None:
    """1銘柄分のテクニカル指標とスコアを算出する。データ不足ならNoneを返す。"""
    end = datetime.today()
    start = end - timedelta(days=period_days + 30)  # 余裕を持って取得

    try:
        df = yf.download(ticker, start=start.strftime("%Y-%m-%d"),
                         end=end.strftime("%Y-%m-%d"), progress=False, auto_adjust=False)
    except Exception as e:
        print(f"  [スキップ] {ticker}: 取得エラー ({e})")
        return None

    if df.empty or len(df) < min_data_points:
        print(f"  [スキップ] {ticker}: データ不足 ({len(df) if not df.empty else 0} 日)")
        return None

    # MultiIndexカラム対応（yfinanceのバージョン差を吸収）
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"].astype(float)
    volume = df["Volume"].astype(float)

    # 各種指標
    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(min(200, len(close))).mean()
    rsi14 = compute_rsi(close, 14)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std

    last_close = float(close.iloc[-1])
    last_sma20 = float(sma20.iloc[-1]) if not np.isnan(sma20.iloc[-1]) else np.nan
    last_sma50 = float(sma50.iloc[-1]) if not np.isnan(sma50.iloc[-1]) else np.nan
    last_sma200 = float(sma200.iloc[-1]) if not np.isnan(sma200.iloc[-1]) else np.nan
    last_rsi = float(rsi14.iloc[-1]) if not np.isnan(rsi14.iloc[-1]) else np.nan

    # モメンタム（リターン）
    ret_20d = (last_close / float(close.iloc[-21]) - 1) * 100 if len(close) > 21 else np.nan
    ret_60d = (last_close / float(close.iloc[-61]) - 1) * 100 if len(close) > 61 else np.nan

    # 出来高トレンド: 直近5日平均 vs 過去20日平均
    vol_recent = float(volume.tail(5).mean())
    vol_base = float(volume.tail(20).mean())
    vol_ratio = vol_recent / vol_base if vol_base > 0 else 1.0

    # ボリンジャー位置（0=下限, 1=上限）
    bb_pos = np.nan
    if not np.isnan(bb_upper.iloc[-1]) and not np.isnan(bb_lower.iloc[-1]):
        rng = float(bb_upper.iloc[-1]) - float(bb_lower.iloc[-1])
        if rng > 0:
            bb_pos = (last_close - float(bb_lower.iloc[-1])) / rng

    # ===== スコアリング（合計 100点満点）=====
    score = 0.0
    reasons = []

    # 1. トレンド: 終値 > SMA20 > SMA50 > SMA200 (各10点, 最大30点)
    if not np.isnan(last_sma20) and last_close > last_sma20:
        score += 10
        reasons.append("終値>SMA20")
    if not np.isnan(last_sma50) and not np.isnan(last_sma20) and last_sma20 > last_sma50:
        score += 10
        reasons.append("SMA20>SMA50")
    if not np.isnan(last_sma200) and not np.isnan(last_sma50) and last_sma50 > last_sma200:
        score += 10
        reasons.append("SMA50>SMA200")

    # 2. ゴールデンクロス（直近10営業日以内）: 15点
    if len(sma20.dropna()) >= 11 and len(sma50.dropna()) >= 11:
        recent20 = sma20.tail(11).values
        recent50 = sma50.tail(11).values
        crossed = False
        for i in range(1, len(recent20)):
            if recent20[i - 1] <= recent50[i - 1] and recent20[i] > recent50[i]:
                crossed = True
                break
        if crossed:
            score += 15
            reasons.append("直近ゴールデンクロス")

    # 3. RSI: 売られすぎ反発(30〜45)で+15点、適度(45〜65)で+10点、過熱(>75)は減点
    if not np.isnan(last_rsi):
        if 30 <= last_rsi < 45:
            score += 15
            reasons.append(f"RSI反発圏({last_rsi:.0f})")
        elif 45 <= last_rsi <= 65:
            score += 10
            reasons.append(f"RSI健全({last_rsi:.0f})")
        elif last_rsi > 75:
            score -= 10
            reasons.append(f"RSI過熱({last_rsi:.0f})")

    # 4. モメンタム: 20日リターン(最大15点)、60日リターン(最大10点)
    if not np.isnan(ret_20d):
        score += max(min(ret_20d, 15), -10) * (15 / 15)  # 簡易マッピング
        if ret_20d > 0:
            reasons.append(f"20日+{ret_20d:.1f}%")
    if not np.isnan(ret_60d):
        score += max(min(ret_60d / 2, 10), -5)
        if ret_60d > 0:
            reasons.append(f"60日+{ret_60d:.1f}%")

    # 5. 出来高トレンド: 直近5日平均が20日平均より高ければ+5、1.5倍以上で+10
    if vol_ratio >= 1.5:
        score += 10
        reasons.append(f"出来高急増(x{vol_ratio:.1f})")
    elif vol_ratio >= 1.1:
        score += 5
        reasons.append(f"出来高増(x{vol_ratio:.2f})")

    # 6. ボリンジャーバンド: 中央付近(0.3〜0.7)で安定上昇なら+5、上限超え(>1.0)は減点
    if not np.isnan(bb_pos):
        if 0.3 <= bb_pos <= 0.7:
            score += 5
        elif bb_pos > 1.0:
            score -= 5
            reasons.append("BB上抜け(過熱)")

    return {
        "ticker": ticker,
        "name": name,
        "終値": round(last_close, 2),
        "SMA20": round(last_sma20, 2) if not np.isnan(last_sma20) else None,
        "SMA50": round(last_sma50, 2) if not np.isnan(last_sma50) else None,
        "SMA200": round(last_sma200, 2) if not np.isnan(last_sma200) else None,
        "RSI14": round(last_rsi, 1) if not np.isnan(last_rsi) else None,
        "20日リターン%": round(ret_20d, 2) if not np.isnan(ret_20d) else None,
        "60日リターン%": round(ret_60d, 2) if not np.isnan(ret_60d) else None,
        "出来高比": round(vol_ratio, 2),
        "BB位置": round(bb_pos, 2) if not np.isnan(bb_pos) else None,
        "スコア": round(score, 1),
        "シグナル": " / ".join(reasons) if reasons else "-",
    }


print("テクニカル分析の関数を定義しました。")


In [ ]:
# ===== 候補銘柄を一括分析し、上昇期待度ランキングを作成 =====
print(f"{len(candidate_tickers)} 銘柄の過去 {lookback_days} 日分のデータを分析します...\n")

results = []
for tkr, nm in candidate_tickers.items():
    print(f"  分析中: {tkr} ({nm})")
    r = analyze_ticker(tkr, nm, lookback_days)
    if r is not None:
        results.append(r)

if not results:
    raise RuntimeError("❌ 分析可能なデータが取得できませんでした。銘柄リストやネットワークを確認してください。")

# DataFrameに集約し、スコアで降順ソート
ranking_df = pd.DataFrame(results).sort_values("スコア", ascending=False).reset_index(drop=True)
ranking_df.index = ranking_df.index + 1  # 1始まり
ranking_df.index.name = "順位"

print("\n" + "=" * 70)
print(f"📈 上昇期待度ランキング TOP {top_n}")
print("=" * 70)

top_df = ranking_df.head(top_n)
for rank, row in top_df.iterrows():
    print(f"\n第{rank}位: {row['ticker']} ({row['name']})  スコア: {row['スコア']}")
    print(f"  終値 {row['終値']} / RSI {row['RSI14']} / 20日 {row['20日リターン%']}% / 60日 {row['60日リターン%']}%")
    print(f"  シグナル: {row['シグナル']}")

print("\n" + "=" * 70)
print("全銘柄のスコア表:")
print("=" * 70)
display_cols = ["ticker", "name", "スコア", "終値", "RSI14", "20日リターン%", "60日リターン%", "出来高比", "シグナル"]
print(ranking_df[display_cols].to_string())

# CSV出力
recommend_filename = f"recommend_top_{datetime.today().strftime('%Y%m%d')}.csv"
try:
    ranking_df.to_csv(recommend_filename, encoding="utf-8-sig")
    print(f"\n✓ ランキングをCSVに保存しました: {recommend_filename}")
    try:
        from google.colab import files as colab_files
        colab_files.download(recommend_filename)
    except Exception:
        print("  (Colab以外の環境ではダウンロードはスキップされます)")
except Exception as e:
    print(f"⚠️ CSV保存に失敗: {e}")

print("\n⚠️ 免責事項: このスコアは過去データからのテクニカル分析であり、将来の値上がりを保証するものではありません。")
print("   投資判断はご自身の責任で行ってください。")

# ----- スコアのバーチャート -----
fig, axes2 = plt.subplots(1, 2, figsize=(16, max(4, len(ranking_df) * 0.5 + 1)))

# 左: スコアランキング（横棒グラフ）
ax_bar = axes2[0]
labels = [f"{row['ticker']} ({row['name']})" for _, row in ranking_df.iterrows()]
scores_list = ranking_df['スコア'].tolist()
bar_colors = ['#1565C0' if i < top_n else '#B0BEC5' for i in range(len(ranking_df))]
hbars = ax_bar.barh(labels, scores_list, color=bar_colors, edgecolor='white', height=0.65)
for bar, sc in zip(hbars, scores_list):
    ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{sc:.1f}', va='center', fontsize=9)
ax_bar.set_xlabel('スコア', fontsize=11)
ax_bar.set_title('上昇期待度スコア ランキング', fontsize=12)
ax_bar.invert_yaxis()
ax_bar.set_xlim(0, max(scores_list) * 1.18 if max(scores_list) > 0 else 10)
ax_bar.axvline(50, color='orange', linestyle='--', linewidth=0.9, alpha=0.7)
ax_bar.grid(axis='x', alpha=0.3)

# 右: 上位銘柄のテクニカル指標ヒートマップ
ax_heat = axes2[1]
heat_cols = ['RSI14', '20日リターン%', '60日リターン%', '出来高比']
heat_labels = ['RSI14', '20日\nリターン%', '60日\nリターン%', '出来高比']
top_data = top_df[heat_cols].copy()
top_data.index = [row['ticker'] for _, row in top_df.iterrows()]
top_data = top_data.astype(float).fillna(0)
_norm = top_data.copy()
for col in _norm.columns:
    mn, mx = _norm[col].min(), _norm[col].max()
    _norm[col] = (_norm[col] - mn) / (mx - mn + 1e-9)
im = ax_heat.imshow(_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax_heat.set_xticks(range(len(heat_labels)))
ax_heat.set_xticklabels(heat_labels, fontsize=9)
ax_heat.set_yticks(range(len(top_data.index)))
ax_heat.set_yticklabels(top_data.index, fontsize=9)
ax_heat.set_title(f'TOP {top_n} テクニカル指標ヒートマップ', fontsize=12)
for i in range(len(top_data.index)):
    for j in range(len(heat_cols)):
        val = top_data.iloc[i, j]
        nv = _norm.iloc[i, j]
        ax_heat.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9,
                     color='black' if 0.2 < nv < 0.8 else 'white')
plt.colorbar(im, ax=ax_heat, shrink=0.8, label='相対スコア')

plt.tight_layout()
plt.savefig('recommend_chart.png', dpi=130, bbox_inches='tight')
plt.show()
print("\n✓ レコメンドチャートを保存しました: recommend_chart.png")
